# Corti API Workflow Demo
## Audio → Transcript → Facts → SOAP Document

This notebook demonstrates the complete workflow of processing medical audio through the Corti API to generate a structured SOAP document.

**Each step is broken down for easy testing, troubleshooting, and showcasing.**

---
## Setup & Authentication

Initialize the Corti API client and authenticate.

In [ ]:
from dotenv import load_dotenv
from corti_client import CortiClient
import requests
import time
import uuid
import json
import os

In [ ]:

# Load credentials
load_dotenv()

# Initialize and authenticate
client = CortiClient()
client.authenticate()

print("✓ Successfully authenticated with Corti API")
print(f"Environment: {client.environment}")
print(f"Tenant: {client.tenant_name}")

---
## Assumptions & Design Decisions

### API Assumptions:
- OAuth2 token-based authentication with client credentials
- Interaction lifecycle management required for all operations
- Asynchronous transcription processing with polling mechanism
- Primary language: English (en-US)

### Edge Cases Handled:
- **Transcription timeout**: Maximum wait time of 300 seconds with 3-second polling intervals
- **Large audio files**: Upload timeout set to 60 seconds
- **Status checking**: Handles 'processing', 'completed', and 'failed' states
- **Response parsing**: Robust extraction of nested JSON structures

### Implementation Choices:
- **Diarization enabled**: For speaker separation in transcripts
- **Global sequential mode**: For document generation consistency
- **Guardrails enabled**: For safe, compliant output
- **Structured facts**: Using `{text, group, source}` format for compatibility

---
# Step 1: Audio → Transcript

Breaking down the transcription process into individual steps:
1. Create interaction
2. Upload audio recording
3. Request transcription
4. Poll for completion
5. Retrieve transcript
6. Extract text
7. Save to file

### Step 1.1: Create Interaction
**API**: `POST /interactions`

Create a new interaction session with patient and encounter information.

In [4]:
headers = client.get_headers(include_tenant=True)

# Create interaction payload
interaction_payload = {
    "encounter": {
        "identifier": str(uuid.uuid4()),
        "status": "planned",
        "type": "first_consultation"
    },
    "patient": {
        "identifier": str(uuid.uuid4()),
        "gender": "unknown"
    }
}

print("Creating interaction...")
response = requests.post(
    f"{client.api_url}/interactions",
    headers=headers,
    json=interaction_payload,
    timeout=30
)
response.raise_for_status()
interaction_data = response.json()
interaction_id = interaction_data["interactionId"]

print(f"✓ Interaction created successfully!")
print(f"Interaction ID: {interaction_id}")
print(f"\nSample response:")
print(json.dumps(interaction_data, indent=2)[:500] + "...")

### Step 1.2: Upload Audio File
**API**: `POST /interactions/{id}/recordings/`

Upload the audio file to the interaction.

In [5]:
# Specify audio file path
audio_path = "data/samples/TalkCPR- A real patient and doctor interaction filmed.mp3"

print(f"Uploading audio file: {audio_path}")
print(f"File size: {os.path.getsize(audio_path) / (1024*1024):.2f} MB\n")

with open(audio_path, 'rb') as audio_file:
    files = {'file': audio_file}
    upload_headers = {
        "Authorization": f"Bearer {client.access_token}",
        "Tenant-Name": client.tenant_name
    }

    response = requests.post(
        f"{client.api_url}/interactions/{interaction_id}/recordings/",
        headers=upload_headers,
        files=files,
        timeout=60
    )
    response.raise_for_status()
    recording_data = response.json()
    recording_id = recording_data["recordingId"]

print(f"✓ Audio uploaded successfully!")
print(f"Recording ID: {recording_id}")

### Step 1.3: Request Transcription
**API**: `POST /interactions/{id}/transcripts`

Submit a transcription request for the uploaded audio.

In [6]:
# Transcription configuration
transcript_payload = {
    "recordingId": recording_id,
    "primaryLanguage": "en",
    "isDictation": False,
    "isMultichannel": False,
    "diarize": True
}


print("Requesting transcription...")
print(f"Configuration: {json.dumps(transcript_payload, indent=2)}\n")

response = requests.post(
    f"{client.api_url}/interactions/{interaction_id}/transcripts",
    headers=headers,
    json=transcript_payload,
    timeout=30
)
response.raise_for_status()
transcript_data = response.json()
transcript_id = transcript_data["id"]

print(f"✓ Transcription requested successfully!")
print(f"Transcript ID: {transcript_id}")

### Step 1.4: Poll for Transcription Status
Wait for the transcription to complete by polling the status endpoint.

In [7]:
print("Polling for transcription completion...")
print("(This may take 30-120 seconds depending on audio length)\n")

max_wait_seconds = 300
poll_interval = 3
start_time = time.time()

while time.time() - start_time < max_wait_seconds:
    # Check status
    response = requests.get(
        f"{client.api_url}/interactions/{interaction_id}/transcripts/{transcript_id}/status",
        headers=headers,
        timeout=10
    )
    response.raise_for_status()
    status_data = response.json()
    status = status_data.get('status', 'processing')

    elapsed = int(time.time() - start_time)
    print(f"[{elapsed}s] Status: {status}")

    if status == 'completed':
        print("\n✓ Transcription completed!")
        break
    elif status == 'failed':
        error_msg = status_data.get('error', 'Unknown error')
        raise Exception(f"Transcript processing failed: {error_msg}")

    time.sleep(poll_interval)
else:
    raise TimeoutError(f"Transcript did not complete within {max_wait_seconds} seconds")

print(f"\nFinal status response:")
print(json.dumps(status_data, indent=2))

### Step 1.5: Retrieve Full Transcript
Get the complete transcript data from the API.

In [10]:
print("Retrieving full transcript...")

response = requests.get(
    f"{client.api_url}/interactions/{interaction_id}/transcripts/{transcript_id}",
    headers=headers,
    timeout=50
)
response.raise_for_status()
transcript_result = response.json()

print("✓ Transcript retrieved successfully!")
print(f"\nTranscript contains {len(transcript_result.get('transcripts', []))} segments")
print(f"\nFull transcript structure:")
print(json.dumps(transcript_result, indent=2)[:1000] + "..." if len(json.dumps(transcript_result)) > 1000 else json.dumps(transcript_result, indent=2))

### Step 1.6: Extract Transcript Text
**Result**: Final transcribed text ready for fact extraction

In [11]:
# Extract text from transcript segments
transcript_parts = []

if 'transcripts' in transcript_result:
    for entry in transcript_result['transcripts']:
        if 'text' in entry:
            transcript_parts.append(entry['text'])

transcript_text = " ".join(transcript_parts)

print("TRANSCRIPT TEXT:")
print("=" * 70)
print(transcript_text)
print("=" * 70)
print(f"\nLength: {len(transcript_text)} characters")
print(f"Word count: ~{len(transcript_text.split())} words")

### Step 1.7: Save Transcript to File
Save the transcript text to the data/transcripts directory.

In [12]:
folder_name = "data/transcripts"
os.makedirs(folder_name, exist_ok=True)

file_path = os.path.join(folder_name, f"{interaction_id}_transcript.txt")

with open(file_path, "w", encoding="utf-8") as f:
    f.write(transcript_text)

print(f"✓ Transcript saved to: {file_path}")
print(f"File size: {os.path.getsize(file_path)} bytes")

---
# Step 2: Transcript → Facts

Breaking down the fact extraction process:
1. Prepare context payload
2. Send extract-facts request
3. Display extracted facts
4. Save facts to file

### Step 2.1: Prepare Context Payload
Format the transcript text for the extract-facts API.

In [13]:
output_language = "en-US"

facts_payload = {
    "context": [
        {
            "type": "text",
            "text": transcript_text
        }
    ],
    "outputLanguage": output_language
}

print("Facts extraction payload prepared:")
print(json.dumps(facts_payload, indent=2)[:500] + "..." if len(json.dumps(facts_payload)) > 500 else json.dumps(facts_payload, indent=2))

### Step 2.2: Send Extract-Facts Request
**API**: `POST /tools/extract-facts`

Call the Corti API to extract structured facts from the transcript.

In [14]:
print("Extracting facts from transcript...")

response = requests.post(
    f"{client.api_url}/tools/extract-facts",
    headers=headers,
    json=facts_payload,
    timeout=30
)
response.raise_for_status()
facts_result = response.json()

print(f"✓ Facts extracted successfully!")
print(f"Number of facts: {len(facts_result.get('facts', []))}")

### Step 2.3: Display Extracted Facts
Show the facts in a readable format.

In [15]:
facts_list = facts_result.get('facts', [])

print("EXTRACTED FACTS:")
print("=" * 70)
for i, fact in enumerate(facts_list, 1):
    # Handle both string and structured fact formats
    if isinstance(fact, dict):
        fact_text = fact.get('text', str(fact))
        fact_group = fact.get('group', 'N/A')
        fact_source = fact.get('source', 'N/A')
        print(f"{i}. [{fact_group}] {fact_text}")
    else:
        print(f"{i}. {fact}")
print("=" * 70)
print(f"\nTotal facts: {len(facts_list)}")

### Step 2.4: Save Facts to File
Save the extracted facts to the data/facts directory.

In [16]:
folder_name = "data/facts"
os.makedirs(folder_name, exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
file_path = os.path.join(folder_name, f"facts_{timestamp}.json")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(facts_result, f, indent=2, ensure_ascii=False)

print(f"✓ Facts saved to: {file_path}")
print(f"File size: {os.path.getsize(file_path)} bytes")

---
# Step 3: Facts → SOAP Document

Breaking down the SOAP document generation:
1. Format facts for API
2. Create document generation request (returns complete document)
3. Display SOAP document structure
4. Display SOAP sections
5. Save document to file

### Step 3.1: Format Facts for API
Convert the facts list into the format required by the documents API.

In [17]:
# Facts are already properly structured from extract_facts API
# Each fact should have: text, group, source
# If facts are strings (legacy), convert them; otherwise use as-is
formatted_facts = []
for fact in facts_list:
    if isinstance(fact, str):
        # Legacy support: convert string to structured fact
        formatted_facts.append({
            "text": fact,
            "group": "others",
            "source": "core"
        })
    elif isinstance(fact, dict):
        # Use structured fact as-is (already has text, group, source)
        formatted_facts.append(fact)

print(f"Formatted {len(formatted_facts)} facts for document generation")
print("\nSample formatted fact:")
print(json.dumps(formatted_facts[0] if formatted_facts else {}, indent=2))
print("\nAll formatted facts (first 500 chars):")
print(json.dumps(formatted_facts, indent=2)[:500] + "..." if len(json.dumps(formatted_facts)) > 500 else json.dumps(formatted_facts, indent=2))

### Step 3.2: Create Document Generation Request
**API**: `POST /interactions/{id}/documents/`

Submit a request to generate a SOAP document from the facts.

In [ ]:
document_payload = {
    "context": [
        {
            "type": "facts",
            "data": formatted_facts
        }
    ],
    "templateKey": "corti-soap",
    "outputLanguage": output_language,
    "name": "SOAP Note",
    "disableGuardrails": False,
    "documentationMode": "global_sequential"
}

print("Creating SOAP document generation request...")
print(f"Template: {document_payload['templateKey']}")
print(f"Language: {document_payload['outputLanguage']}\n")

response = requests.post(
    f"{client.api_url}/interactions/{interaction_id}/documents/",
    headers=headers,
    json=document_payload,
    timeout=60
)
response.raise_for_status()
soap_document = response.json()

print(f"✓ SOAP document generated!")
print(f"Document ID: {soap_document.get('id', 'N/A')}")

### Step 3.4: Retrieve Complete SOAP Document
View the full SOAP document structure.

In [20]:
print("SOAP Document Structure:")
print("=" * 70)
print(json.dumps(soap_document, indent=2))
print("=" * 70)

### Step 3.5: Display SOAP Sections
**Result**: Complete clinical documentation in SOAP format

In [31]:
print("SOAP DOCUMENT:")
print("=" * 70)

if 'sections' in soap_document.keys():
    for chunk in soap_document['sections']:
        section = chunk.get('name')
        content = chunk.get('text', '')
        print(f"\n{section}:")
        print("-" * 70)
        print(content)
else:
    print("No chunks found in document")

print("\n" + "=" * 70)

### Step 3.6: Save SOAP Document to File
Save the complete SOAP document to the data/SOAP directory.

In [ ]:
folder_name = "data/soap_documents"
os.makedirs(folder_name, exist_ok=True)

file_path = os.path.join(folder_name, f"{interaction_id}_soap.json")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(soap_document, f, indent=2, ensure_ascii=False)

print(f"✓ SOAP document saved to: {file_path}")
print(f"File size: {os.path.getsize(file_path)} bytes")

---
# Summary

## Complete Workflow Recap

### Step 1: Audio → Transcript (7 steps)
1. ✓ Created interaction
2. ✓ Uploaded audio file
3. ✓ Requested transcription
4. ✓ Polled for completion
5. ✓ Retrieved transcript
6. ✓ Extracted text
7. ✓ Saved to file

### Step 2: Transcript → Facts (4 steps)
1. ✓ Prepared context payload
2. ✓ Sent extract-facts request
3. ✓ Displayed facts
4. ✓ Saved to file

### Step 3: Facts → SOAP (5 steps)
1. ✓ Formatted facts for API
2. ✓ Created document request (returns complete document)
3. ✓ Displayed document structure
4. ✓ Displayed SOAP sections
5. ✓ Saved to file

### Output Files
- Transcript: `data/transcripts/{interaction_id}_transcript.txt`
- Facts: `data/facts/facts_{timestamp}.json`
- SOAP: `data/SOAP/{interaction_id}_soap.json`

---

**Benefits of this step-by-step approach:**
- ✅ Each step is independently executable
- ✅ Easy to identify where issues occur
- ✅ Can rerun specific steps without starting over
- ✅ Clear visibility into API requests and responses
- ✅ Perfect for demos and presentations